In [1]:
%pip install pandas numpy scikit-learn matplotlib seaborn imbalanced-learn xgboost lightgbm folium --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, accuracy_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries loaded successfully.')

In [ ]:
df = pd.read_csv('../data/collisions_clean.csv')
df['CRASH DATE'] = pd.to_datetime(df['CRASH DATE'], errors='coerce')
df['hour'] = pd.to_numeric(df['hour'], errors='coerce')

print(f'Dataset loaded: {len(df):,} records')
print(f'Date range: {df["CRASH DATE"].min().date()} to {df["CRASH DATE"].max().date()}')

In [ ]:
# Feature engineering (matches submission.ipynb)
df['target_severity'] = df['severity'].apply(lambda x: 1 if x in ['Injury', 'Fatal'] else 0)
df['BOROUGH'] = df['BOROUGH'].fillna('Unknown')
df['vehicle_type_clean'] = df['vehicle_type_clean'].fillna('Unknown')
df['factor_category'] = df['factor_category'].fillna('Unknown')
df['season'] = df['season'].fillna('Unknown')

def time_interval(hour):
    if pd.isna(hour): return 'Unknown'
    hour = int(hour)
    if 6 <= hour <= 9:   return 'Morning Rush'
    elif 10 <= hour <= 15: return 'Midday'
    elif 16 <= hour <= 19: return 'Evening Rush'
    elif 20 <= hour <= 23: return 'Night'
    else: return 'Late Night'

df['time_interval'] = df['hour'].apply(time_interval)
df['day_of_week'] = df['CRASH DATE'].dt.dayofweek

df_encoded = pd.get_dummies(
    df, columns=['BOROUGH', 'vehicle_type_clean', 'factor_category', 'season'],
    drop_first=True
)

temporal_cutoff = pd.to_datetime('2024-01-01')
train_df = df_encoded[df_encoded['CRASH DATE'] < temporal_cutoff].copy()
test_df  = df_encoded[df_encoded['CRASH DATE'] >= temporal_cutoff].copy()

exclude = [
    'CRASH DATE','severity','target_severity','total_casualties',
    'has_injury','has_fatality','has_pedestrian_casualty','has_cyclist_casualty',
    'NUMBER OF PERSONS INJURED','NUMBER OF PERSONS KILLED',
    'NUMBER OF PEDESTRIANS INJURED','NUMBER OF PEDESTRIANS KILLED',
    'NUMBER OF CYCLIST INJURED','NUMBER OF CYCLIST KILLED',
    'NUMBER OF MOTORIST INJURED','NUMBER OF MOTORIST KILLED',
    'LATITUDE','LONGITUDE','primary_factor'
]

feature_cols = [
    c for c in df_encoded.columns
    if c not in exclude and df_encoded[c].dtype in ['int64','float64','uint8','bool']
]

X_train = train_df[feature_cols].fillna(0)
X_test  = test_df[feature_cols].fillna(0)
y_train = train_df['target_severity']
y_test  = test_df['target_severity']

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Positive rate — Train: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%')

In [ ]:
# Train Logistic Regression (final model, EMS priority)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# Train LightGBM + SMOTE (best AUC from M6)
smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_train, y_train)
lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.1, random_state=42, verbose=-1)
lgbm.fit(X_sm, y_sm)
y_pred_lgbm = lgbm.predict(X_test)
y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]

# Feature importance dataframe
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lr_model.coef_[0]})
coef_df = coef_df.sort_values('Coefficient', key=abs, ascending=False)

print('Models trained.')
print(f'LR   — Recall: {recall_score(y_test, y_pred_lr):.3f} | AUC: {roc_auc_score(y_test, y_prob_lr):.3f}')
print(f'LGBM — Recall: {recall_score(y_test, y_pred_lgbm):.3f} | AUC: {roc_auc_score(y_test, y_prob_lgbm):.3f}')

In [ ]:
# Heatmap 1: Borough x Time Interval
df_plot = df[df['BOROUGH'] != 'Unknown'].copy()
time_order = ['Late Night', 'Morning Rush', 'Midday', 'Evening Rush', 'Night']

borough_time = df_plot.groupby(['BOROUGH', 'time_interval'])['target_severity'].mean().unstack() * 100
borough_time = borough_time[[t for t in time_order if t in borough_time.columns]]

plt.figure(figsize=(12, 5))
sns.heatmap(
    borough_time, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Injury Rate (%)'}
)
plt.title('Heatmap 1: Injury Rate (%) by Borough x Time of Day', fontsize=14, fontweight='bold')
plt.xlabel('Time Interval', fontsize=12)
plt.ylabel('Borough', fontsize=12)
plt.xticks(rotation=20)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap 2: Hour x Day of Week
df_hm = df[df['hour'].notna()].copy()
df_hm['hour_int'] = df_hm['hour'].astype(int)

hour_day = df_hm.groupby(['day_of_week', 'hour_int'])['target_severity'].mean().unstack() * 100
day_labels = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

plt.figure(figsize=(18, 6))
sns.heatmap(
    hour_day, cmap='YlOrRd', linewidths=0.2, linecolor='white',
    cbar_kws={'label': 'Injury Rate (%)'},
    xticklabels=[str(h) for h in range(24)]
)
plt.title('Heatmap 2: Injury Rate (%) by Hour x Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Hour of Day', fontsize=12)
plt.ylabel('Day of Week', fontsize=12)
plt.yticks(ticks=[i+0.5 for i in range(7)], labels=day_labels, rotation=0)
plt.tight_layout()
plt.show()

print('Key insight: Late-night hours (0-4 AM) consistently show the highest injury rates.')

In [ ]:
# Heatmap 3: Borough x Contributing Factor
df_bfac = df[(df['BOROUGH'] != 'Unknown') & (df['factor_category'] != 'Unknown')].copy()
top_factors = df_bfac['factor_category'].value_counts().head(8).index.tolist()
df_bfac = df_bfac[df_bfac['factor_category'].isin(top_factors)]

boro_factor = df_bfac.groupby(['BOROUGH', 'factor_category'])['target_severity'].mean().unstack() * 100
boro_factor = boro_factor[top_factors]

plt.figure(figsize=(14, 5))
sns.heatmap(
    boro_factor, annot=True, fmt='.1f', cmap='YlOrRd',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Injury Rate (%)'}
)
plt.title('Heatmap 3: Injury Rate (%) by Borough x Contributing Factor', fontsize=14, fontweight='bold')
plt.xlabel('Contributing Factor Category', fontsize=12)
plt.ylabel('Borough', fontsize=12)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap 4: Geographic risk map (NYC scatter)
df_geo = df.dropna(subset=['LATITUDE', 'LONGITUDE']).copy()
df_geo = df_geo[
    df_geo['LATITUDE'].between(40.4, 40.95) &
    df_geo['LONGITUDE'].between(-74.3, -73.65)
]
df_sample = df_geo.sample(n=min(60000, len(df_geo)), random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

axes[0].scatter(df_sample['LONGITUDE'], df_sample['LATITUDE'], alpha=0.03, s=1, c='steelblue')
axes[0].set_title('All Collisions — Geographic Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_facecolor('#1a1a2e')

no_inj = df_sample[df_sample['target_severity'] == 0]
inj    = df_sample[df_sample['target_severity'] == 1]
axes[1].scatter(no_inj['LONGITUDE'], no_inj['LATITUDE'], alpha=0.03, s=1, c='#4ECDC4')
axes[1].scatter(inj['LONGITUDE'],    inj['LATITUDE'],    alpha=0.05, s=1, c='#FF6B6B')
axes[1].set_title('Injury vs. No-Injury — Geographic Risk', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].set_facecolor('#1a1a2e')
legend_patches = [
    mpatches.Patch(color='#FF6B6B', label='Injury / Fatal'),
    mpatches.Patch(color='#4ECDC4', label='Property Damage Only')
]
axes[1].legend(handles=legend_patches, loc='lower right', fontsize=10)

plt.suptitle('Heatmap 4: NYC Collision Risk Map', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()
print(f'Plotted {len(df_sample):,} collisions — {len(inj):,} injuries (red), {len(no_inj):,} property-damage-only (teal).')

In [ ]:
# (injury collisions only)
try:
    import folium
    from folium.plugins import HeatMap
    import os

    os.makedirs('../outputs', exist_ok=True)

    df_folium = df_geo[df_geo['target_severity'] == 1].sample(n=min(30000, len(df_geo)), random_state=42)
    nyc_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11, tiles='CartoDB dark_matter')
    HeatMap(
        data=df_folium[['LATITUDE', 'LONGITUDE']].values.tolist(),
        radius=8, blur=6, max_zoom=13,
        gradient={0.2: 'blue', 0.5: 'yellow', 1.0: 'red'}
    ).add_to(nyc_map)
    nyc_map.save('../outputs/nyc_injury_heatmap.html')
    print('Interactive heatmap saved: ../outputs/nyc_injury_heatmap.html')
    print('Open in a browser to explore interactively.')
    display(nyc_map)
except Exception as e:
    print(f'Folium map skipped ({e}) — static maps above are the primary deliverable.')

In [ ]:
# Panel 1: Model performance comparison
model_results = {
    'LR (balanced)':  {
        'Accuracy':  accuracy_score(y_test, y_pred_lr),
        'Precision': precision_score(y_test, y_pred_lr),
        'Recall':    recall_score(y_test, y_pred_lr),
        'F1':        f1_score(y_test, y_pred_lr),
        'ROC AUC':   roc_auc_score(y_test, y_prob_lr),
    },
    'LightGBM+SMOTE': {
        'Accuracy':  accuracy_score(y_test, y_pred_lgbm),
        'Precision': precision_score(y_test, y_pred_lgbm),
        'Recall':    recall_score(y_test, y_pred_lgbm),
        'F1':        f1_score(y_test, y_pred_lgbm),
        'ROC AUC':   roc_auc_score(y_test, y_prob_lgbm),
    },
}
results_df = pd.DataFrame(model_results).T

fig, axes = plt.subplots(1, 5, figsize=(18, 5))
colors = ['#4ECDC4', '#FF6B6B']

for i, metric in enumerate(['Accuracy','Precision','Recall','F1','ROC AUC']):
    vals = results_df[metric].values
    bars = axes[i].bar(results_df.index, vals, color=colors, edgecolor='black', alpha=0.85)
    axes[i].set_title(metric, fontsize=13, fontweight='bold')
    axes[i].set_ylim(0, 1.08)
    axes[i].set_xticklabels(results_df.index, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                     f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
    axes[i].grid(axis='y', alpha=0.3)

plt.suptitle('Dashboard Panel 1: Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print(results_df.round(4).to_string())

In [ ]:
# Panel 2: Confusion Matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_lr,   'Logistic Regression (balanced)'),
    (axes[1], y_pred_lgbm, 'LightGBM + SMOTE'),
]:
    cm = confusion_matrix(y_test, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    annot = np.array([[f'{cm[r,c]:,}\n({cm_norm[r,c]:.1f}%)' for c in range(2)] for r in range(2)])
    sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                xticklabels=['No Injury','Injury/Fatal'],
                yticklabels=['No Injury','Injury/Fatal'], linewidths=0.5)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')

plt.suptitle('Dashboard Panel 2: Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Panel 3: ROC and Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for y_prob, label, color in [
    (y_prob_lr,   f'LR balanced (AUC={roc_auc_score(y_test, y_prob_lr):.3f})',   '#4ECDC4'),
    (y_prob_lgbm, f'LightGBM+SMOTE (AUC={roc_auc_score(y_test, y_prob_lgbm):.3f})', '#FF6B6B'),
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, linewidth=2.5, label=label, color=color)
ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Random')
ax.set_title('ROC Curves', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

ax = axes[1]
for y_prob, label, color in [
    (y_prob_lr,   f'LR balanced (AP={average_precision_score(y_test, y_prob_lr):.3f})',   '#4ECDC4'),
    (y_prob_lgbm, f'LightGBM+SMOTE (AP={average_precision_score(y_test, y_prob_lgbm):.3f})', '#FF6B6B'),
]:
    prec, rec, _ = precision_recall_curve(y_test, y_prob)
    ax.plot(rec, prec, linewidth=2.5, label=label, color=color)
ax.axhline(y=y_test.mean(), color='grey', linestyle='--', alpha=0.6, label=f'Baseline ({y_test.mean():.2f})')
ax.set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.legend(fontsize=10); ax.grid(alpha=0.3)

plt.suptitle('Dashboard Panel 3: ROC & Precision-Recall Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Panel 4: Top 20 Feature Importances (LR coefficients)
top_n = 20
top_features = coef_df.head(top_n)

plt.figure(figsize=(13, 8))
colors = ['#FF6B6B' if c > 0 else '#4ECDC4' for c in top_features['Coefficient']]
plt.barh(range(top_n), top_features['Coefficient'].values, color=colors, edgecolor='black', alpha=0.85)
plt.yticks(range(top_n), top_features['Feature'].values, fontsize=9)
plt.title('Dashboard Panel 4: Top 20 Features — LR Coefficients', fontsize=14, fontweight='bold')
plt.xlabel('Coefficient  (red = increases injury risk, teal = decreases)', fontsize=11)
plt.axvline(0, color='black', linewidth=0.8)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
legend_patches = [
    mpatches.Patch(color='#FF6B6B', label='Increases injury probability'),
    mpatches.Patch(color='#4ECDC4', label='Decreases injury probability'),
]
plt.legend(handles=legend_patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Panel 5: Prediction score distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_prob, title in [
    (axes[0], y_prob_lr,   'LR (balanced)'),
    (axes[1], y_prob_lgbm, 'LightGBM + SMOTE'),
]:
    ax.hist(y_prob[y_test==0], bins=50, alpha=0.6, color='#4ECDC4', density=True, label='No Injury (actual)')
    ax.hist(y_prob[y_test==1], bins=50, alpha=0.6, color='#FF6B6B', density=True, label='Injury/Fatal (actual)')
    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Default threshold (0.5)')
    ax.set_title(f'{title} — Score Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Probability of Injury')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Dashboard Panel 5: Prediction Score Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Panel 6: Threshold sensitivity for EMS
thresholds = np.linspace(0.1, 0.9, 80)
lr_recalls, lr_precisions, lr_f1s = [], [], []

for t in thresholds:
    pred = (y_prob_lr >= t).astype(int)
    lr_recalls.append(recall_score(y_test, pred, zero_division=0))
    lr_precisions.append(precision_score(y_test, pred, zero_division=0))
    lr_f1s.append(f1_score(y_test, pred, zero_division=0))

plt.figure(figsize=(12, 5))
plt.plot(thresholds, lr_recalls,    color='#FF6B6B', linewidth=2.5, label='Recall (EMS priority)')
plt.plot(thresholds, lr_precisions, color='#4ECDC4', linewidth=2.5, label='Precision')
plt.plot(thresholds, lr_f1s,        color='#FFD700', linewidth=2.5, label='F1 Score')

# Mark EMS threshold (recall >= 0.75)
for t, r in zip(thresholds, lr_recalls):
    if abs(r - 0.75) < 0.012:
        plt.axvline(t, color='grey', linestyle=':', linewidth=1.8)
        plt.text(t+0.01, 0.25, f'EMS threshold\n(recall~0.75)\nt={t:.2f}', fontsize=9, color='grey')
        break

plt.axvline(0.5, color='black', linestyle='--', linewidth=1.2, alpha=0.5, label='Default threshold (0.5)')
plt.title('Dashboard Panel 6: Threshold Sensitivity — LR (balanced)', fontsize=14, fontweight='bold')
plt.xlabel('Classification Threshold', fontsize=12)
plt.ylabel('Score', fontsize=12)
plt.legend(fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Panel 7: Predicted risk vs. actual injury rate by Borough and Time
test_viz = test_df[['CRASH DATE']].copy()
test_viz['BOROUGH']       = df.loc[test_df.index, 'BOROUGH'] if 'BOROUGH' in df.columns else 'Unknown'
test_viz['time_interval'] = df.loc[test_df.index, 'time_interval']
test_viz['target']        = y_test.values
test_viz['pred_prob']     = y_prob_lr
test_viz['BOROUGH']       = test_viz['BOROUGH'].fillna('Unknown')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

boro_risk   = test_viz[test_viz['BOROUGH'] != 'Unknown'].groupby('BOROUGH')['pred_prob'].mean().sort_values()
boro_actual = test_viz[test_viz['BOROUGH'] != 'Unknown'].groupby('BOROUGH')['target'].mean().reindex(boro_risk.index)

ax = axes[0]
x = np.arange(len(boro_risk))
w = 0.35
ax.barh(x - w/2, boro_risk.values,   w, color='#FF6B6B', label='Predicted risk (avg prob)', alpha=0.85, edgecolor='black')
ax.barh(x + w/2, boro_actual.values, w, color='#4ECDC4', label='Actual injury rate',         alpha=0.85, edgecolor='black')
ax.set_yticks(x); ax.set_yticklabels(boro_risk.index)
ax.set_title('Predicted vs. Actual Risk by Borough', fontsize=13, fontweight='bold')
ax.set_xlabel('Rate'); ax.legend(fontsize=10); ax.grid(axis='x', alpha=0.3)

time_order2 = ['Late Night','Morning Rush','Midday','Evening Rush','Night']
time_risk   = test_viz.groupby('time_interval')['pred_prob'].mean().reindex(time_order2).dropna()
time_actual = test_viz.groupby('time_interval')['target'].mean().reindex(time_order2).dropna()

ax = axes[1]
x = np.arange(len(time_risk))
ax.bar(x - w/2, time_risk.values,   w, color='#FF6B6B', label='Predicted risk', alpha=0.85, edgecolor='black')
ax.bar(x + w/2, time_actual.values, w, color='#4ECDC4', label='Actual injury rate', alpha=0.85, edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(time_risk.index, rotation=20, ha='right')
ax.set_title('Predicted vs. Actual Risk by Time Interval', fontsize=13, fontweight='bold')
ax.set_ylabel('Rate'); ax.legend(fontsize=10); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Dashboard Panel 7: Model Predictions vs. Actual Risk', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final Model Card
print('=' * 65)
print('FINAL MODEL CARD')
print('=' * 65)
print('Model:         Logistic Regression (class_weight="balanced")')
print('Task:          Binary classification — Injury/Fatal vs. Property Damage Only')
print('Stakeholder:   Emergency Medical Services (EMS)')
print('Primary goal:  Maximize recall — catch as many injuries as possible')
print()
print('Dataset:       NYC Motor Vehicle Collisions (375,025 records, 2022-2026)')
print('Split:         Temporal — train on 2022-2023, test on 2024+')
print(f'Train size:    {X_train.shape[0]:,} records | {X_train.shape[1]} features')
print(f'Test size:     {X_test.shape[0]:,} records')
print()
print('PERFORMANCE ON TEST SET:')
print(f'  Accuracy:    {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'  Precision:   {precision_score(y_test, y_pred_lr):.4f}')
print(f'  Recall:      {recall_score(y_test, y_pred_lr):.4f}   <- primary metric')
print(f'  F1 Score:    {f1_score(y_test, y_pred_lr):.4f}')
print(f'  ROC AUC:     {roc_auc_score(y_test, y_prob_lr):.4f}')
print()
print('Interpretation:')
print(f'  The model correctly flags {recall_score(y_test, y_pred_lr)*100:.1f}% of all injury collisions.')
print(f'  Of all model alerts, {precision_score(y_test, y_pred_lr)*100:.1f}% are true injuries.')
print()
print('Why LR over tree models?')
print('  - Highest recall (trees have higher AUC but lower recall)')
print('  - Interpretable coefficients for EMS decision-makers')
print('  - Fast inference for real-time shift-level planning')
print('  - Threshold can be lowered to ~0.35 to reach recall >= 0.80')

In [ ]:
# Key Findings
print('=' * 65)
print('KEY FINDINGS')
print('=' * 65)
print()
print('1. TEMPORAL RISK')
print('   - Late Night (0-5 AM) has the HIGHEST injury rate per collision')
print('   - Evening Rush (4-7 PM) has the most total collisions')
print('   - Friday is the most dangerous day of the week')
print('   - EMS implication: increase staffing on Friday late nights')
print()
print('2. SPATIAL RISK')
print('   - Brooklyn leads in total collisions (34.8% of geolocated crashes)')
print('   - Injury rates are consistent across boroughs (~42-44%)')
print('   - Bronx and Staten Island have slightly higher fatality rates')
print()
print('3. TOP PREDICTIVE FEATURES')
top3_pos = coef_df[coef_df['Coefficient'] > 0].head(3)['Feature'].tolist()
top3_neg = coef_df[coef_df['Coefficient'] < 0].head(3)['Feature'].tolist()
print(f'   Strongest injury predictors:  {top3_pos}')
print(f'   Strongest protective factors: {top3_neg}')
print()
print('4. CLASS IMBALANCE')
print('   - 41.7% injury rate — mild imbalance, manageable with class_weight')
print('   - 0.27% fatal-only — too rare for reliable separate prediction')
print('   - SMOTE improved tree model recall but not LR recall')
print()
print('5. EMS DEPLOYMENT RECOMMENDATION')
print('   Use LR (balanced) with threshold = 0.35-0.40')
print('   Target: Late Night Fridays in Bronx and Manhattan')
print('   Expected benefit: catch 78-80% of all injury collisions')

In [ ]:
# Limitations and Future Work
print('=' * 65)
print('LIMITATIONS')
print('=' * 65)
print('''
1. Missing borough data (28.7%) limits spatial granularity
2. No weather/road condition features integrated
3. No street-network features (intersection type, lane count)
4. Fatal-only collisions (0.27%) are too rare for reliable prediction
5. No spatial autocorrelation modeled
''')
print('=' * 65)
print('FUTURE WORK')
print('=' * 65)
print('''
1. Integrate real-time weather API for improved temporal predictions
2. Add OpenStreetMap intersection-level network features
3. Try spatial models (GWR, spatial lag) to capture location clustering
4. Build hourly risk-score feed for live EMS dispatch systems
5. Evaluate fairness across boroughs (equitable resource allocation)
6. Explore LSTM / temporal transformer for sequential crash patterns
''')

In [ ]:
# Project pipeline visualization
fig, ax = plt.subplots(figsize=(15, 4))
ax.axis('off')

steps = [
    ('M1\nEDA', '#AED6F1'),
    ('M2\nData\nCleaning', '#A9DFBF'),
    ('M3\nFeature\nEngineering', '#FAD7A0'),
    ('M4\nBaseline\nModels', '#F1948A'),
    ('M5\nModel\nDevelopment', '#C39BD3'),
    ('M6\nRefinement &\nValidation', '#F9E79F'),
    ('M7\nVisualization\n& Deliverables', '#85C1E9'),
]

width = 0.115
gap = 0.018
start_x = 0.025

for i, (label, color) in enumerate(steps):
    x = start_x + i * (width + gap)
    rect = mpatches.FancyBboxPatch(
        (x, 0.15), width, 0.70,
        boxstyle='round,pad=0.01',
        facecolor=color, edgecolor='#888', linewidth=1.5
    )
    ax.add_patch(rect)
    ax.text(x + width/2, 0.50, label, ha='center', va='center',
            fontsize=9, fontweight='bold', color='#1a1a2e')
    if i < len(steps)-1:
        ax.annotate('', xy=(x + width + gap, 0.50), xytext=(x + width, 0.50),
                    arrowprops=dict(arrowstyle='->', color='grey', lw=1.8))

ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title('NYC Collision Risk Prediction — Full Project Pipeline', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

print('Milestone 7 complete.')